In [ ]:
%matplotlib inline
import os, sys
import pandas as pd
sys.path.insert(0, os.getcwd())  # test.ipynb sits next to dataloader.py / config.py

import matplotlib.pyplot as plt
from dataloader import extract_frame, benchmark_workers
print("ready")

ready


In [24]:
df = pd.DataFrame()
df = extract_frame("6", df) 

I0000 00:00:1784695194.571049  720835 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1784695194.614431  720849 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.173.02), renderer: NVIDIA A100-SXM4-40GB/PCIe/SSE2


landmarker: GPU delegate
source 6: 5030 images (1 process)


6 (serial): 100%|██████████| 5030/5030 [00:30<00:00, 164.14img/s, detected=3712]

source 6: 3712/5030 detected


## Train a simple MLP on `df`

Features per hand = the exact 105-dim vector `app.py` uses:
`normalize_hand_3d(world_xyz)` (63) + image `xy` (42). We train a small MLP and
save it as `asl_mlp.joblib`, which `app.py` loads unchanged.

> Does GPU help? Not here. (1) The net is tiny (105→256→128→classes); CPU trains
> it in seconds and GPU launch/transfer overhead would dominate. (2) `df` is built
> with MediaPipe (TensorFlow) in this kernel, and PyTorch-CUDA in the *same*
> process conflicts with it — it either silently disables CUDA or segfaults the
> kernel. So we use scikit-learn's `MLPClassifier` (CPU): a real simple neural net,
> stable next to MediaPipe, and a drop-in for `app.py` (no torch needed to run it).

In [ ]:
import numpy as np
from tqdm.auto import tqdm

sys.path.insert(0, os.path.join(os.getcwd(), "neural_network"))
from hand_normalization import normalize_hand_3d
from config import KP2D_COLS, KP3D_COLS

# keep detected hands that have valid world 3D landmarks
d = df[(df.detected == 1) & df[KP3D_COLS].notna().all(axis=1)].reset_index(drop=True)
W = d[KP3D_COLS].to_numpy(np.float32).reshape(-1, 21, 3)   # world (for 3D norm)
N = d[KP2D_COLS].to_numpy(np.float32).reshape(-1, 21, 3)   # image (use x,y)

X = np.stack([
    np.concatenate([normalize_hand_3d(w).reshape(-1), n[:, :2].reshape(-1)])
    for w, n in tqdm(zip(W, N), total=len(d), desc="features")
]).astype(np.float32)

y = d.label.to_numpy()                      # string labels; app.py prints these
print("X", X.shape, "| classes:", sorted(set(y)))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
scaler = StandardScaler().fit(Xtr)
Xtr_s, Xva_s = scaler.transform(Xtr), scaler.transform(Xva)

# partial_fit = one epoch per call, so we can show a live progress bar
mlp = MLPClassifier(hidden_layer_sizes=(256, 128), alpha=1e-4,
                    learning_rate_init=1e-3, random_state=0)
classes = np.unique(y)
hist = []
bar = tqdm(range(100), desc="train")
for _ in bar:
    mlp.partial_fit(Xtr_s, ytr, classes=classes)
    acc = mlp.score(Xva_s, yva)
    hist.append(acc)
    bar.set_postfix(loss=f"{mlp.loss_:.3f}", val_acc=f"{acc:.3f}")

print("final val acc:", round(hist[-1], 4))
plt.plot(hist); plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.title("training"); plt.show()

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

# scaler + mlp as one object: app.py feeds it raw 105-dim features, it scales then predicts
clf = Pipeline([("scaler", scaler), ("mlp", mlp)])
print("sanity val acc:", round(clf.score(Xva, yva), 4))

joblib.dump(clf, "asl_mlp.joblib")
print("saved -> asl_mlp.joblib")

## Run it in the webcam app

`asl_mlp.joblib` is a plain scikit-learn `Pipeline` (same interface as the old
`asl_logreg.joblib`), so `app.py` loads it unchanged — just point `--weights` at it:

```bash
python app.py --weights asl_mlp.joblib
```

`app.py` builds the same 105-dim feature per detected hand (`normalize_hand_3d` +
`xy`) and shows the predicted label. (Add `--camera 0` if the default camera index
is wrong.)